In [ ]:
# -------------------------------------------------------------------------
# 0. Dependencies Installation
# -------------------------------------------------------------------------
!pip install -q transformers peft bitsandbytes accelerate scikit-learn seaborn matplotlib


In [ ]:
# -------------------------------------------------------------------------
# 1. Standalone Model 5 Training: RAD-DINO + Qwen-2.5-3B + Completion Prompting
#    + Enhanced 2-Layer Visual Connector + Asymmetric Loss + Multi-Session Resume
# -------------------------------------------------------------------------
import os, sys, gc, glob, time, json, math, random, torch
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    get_cosine_schedule_with_warmup
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    set_peft_model_state_dict, get_peft_model_state_dict
)
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

# Set seed for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training Compute Device: {device}", flush=True)

work_dir = "/kaggle/working"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

# -------------------------------------------------------------------------
# Auto-Locate Data Prep Artifacts (from Published Prep Dataset via + Add Input)
# -------------------------------------------------------------------------
search_dirs = [Path("/kaggle/working"), Path("/kaggle/input")]
manifest_path = None
features_base = None

for base in search_dirs:
    if not base.exists():
        continue
    for root, _, files in os.walk(base):
        if "mimic_master_split_with_rag.csv" in files and manifest_path is None:
            manifest_path = os.path.join(root, "mimic_master_split_with_rag.csv")
            features_base = root
            break
    if manifest_path:
        break

if not manifest_path:
    raise FileNotFoundError(
        "Missing mimic_master_split_with_rag.csv! Please add the Data Prep dataset "
        "via '+ Add Input' -> 'Your Work' / 'Datasets'."
    )

print(f"Found Master Manifest   : {manifest_path}", flush=True)
print(f"Found Features Directory: {features_base}", flush=True)

df_merged = pd.read_csv(manifest_path)
train_df = df_merged[df_merged["split"] == "train"].reset_index(drop=True)
val_df = df_merged[df_merged["split"] == "val"].reset_index(drop=True)
test_df = df_merged[df_merged["split"] == "test"].reset_index(drop=True)

CHEXPERT_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
    "Fracture", "Lung Lesion", "Lung Opacity", "No Finding", "Pleural Effusion",
    "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"
]

# Random sampling of 15,000 diverse training cases while strictly preserving orig_split_idx
max_samples = min(15000, len(train_df))
if max_samples < len(train_df):
    train_sampled_df = train_df.sample(n=max_samples, random_state=42).reset_index(drop=True)
else:
    train_sampled_df = train_df.copy()

print(f"Dataset Cohort: Total Train={len(train_df)} (Sampled={len(train_sampled_df)}) | Val={len(val_df)} | Test={len(test_df)}", flush=True)

# Compute class-specific positive weights for rare class prioritization
pos_counts = train_sampled_df[CHEXPERT_COLS].sum().values
neg_counts = len(train_sampled_df) - pos_counts
pos_weights = np.sqrt(neg_counts / np.clip(pos_counts, a_min=1.0, a_max=None))
pos_weights = np.clip(pos_weights, 1.0, 10.0)
pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float, device=device)
print(f"Calculated ASL Class Weights range: min={pos_weights.min():.2f}, max={pos_weights.max():.2f}", flush=True)

# -------------------------------------------------------------------------
# Qwen-2.5-3B Architecture & LoRA Configuration (r=32, alpha=64)
# -------------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True
)

base_qwen = prepare_model_for_kbit_training(base_qwen)

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
qwen_lora_model = get_peft_model(base_qwen, peft_config)

# -------------------------------------------------------------------------
# Enhanced Model 5 Architecture: Multi-Task + 2-Layer Visual Connector
# -------------------------------------------------------------------------
class MultiTaskQwenModel(nn.Module):
    def __init__(self, qwen_model, visual_dim=768, qwen_dim=2048, num_patches=128):
        super().__init__()
        self.qwen = qwen_model
        self.num_patches = num_patches
        self.qwen_dim = qwen_dim
        
        # Enhanced 2-Layer Visual Connector: 768 -> 2048 -> 2048 with LayerNorm & GELU
        self.visual_projection = nn.Sequential(
            nn.Linear(visual_dim, qwen_dim),
            nn.LayerNorm(qwen_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(qwen_dim, qwen_dim),
            nn.LayerNorm(qwen_dim)
        ).to(device=device)
        
        # Multi-Label Pathology Classifier Head: 768 -> 512 -> 14
        self.classifier = nn.Sequential(
            nn.Linear(visual_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 14)
        ).to(device=device)
        
    def build_4d_prefix_attention_mask(self, batch_size, seq_len, device, dtype, text_pad_mask=None):
        total_len = self.num_patches + seq_len
        mask = torch.full((total_len, total_len), -10000.0, dtype=dtype, device=device)
        # Prefix bidirectional attention among visual patches
        mask[:self.num_patches, :self.num_patches] = 0.0
        # Causal autoregressive attention for text tokens attending to visual prefix + past text
        for i in range(self.num_patches, total_len):
            mask[i, :i+1] = 0.0
            
        mask_4d = mask.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, total_len, total_len).clone()
        if text_pad_mask is not None:
            pad_expanded = text_pad_mask.unsqueeze(1).unsqueeze(2)
            mask_4d[:, :, :, self.num_patches:] = mask_4d[:, :, :, self.num_patches:].masked_fill(pad_expanded == 0, -10000.0)
        return mask_4d
        
    def forward(self, spatial_tokens, mean_features, input_ids, labels=None, text_pad_mask=None):
        batch_size = spatial_tokens.size(0)
        seq_len = input_ids.size(1)
        
        cls_logits = self.classifier(mean_features.float())
        visual_embeds = self.visual_projection(spatial_tokens.float())
        text_embeds = self.qwen.get_input_embeddings()(input_ids)
        
        if visual_embeds.dtype != text_embeds.dtype:
            visual_embeds = visual_embeds.to(text_embeds.dtype)
            
        inputs_embeds = torch.cat([visual_embeds, text_embeds], dim=1)
        attention_mask_4d = self.build_4d_prefix_attention_mask(
            batch_size, seq_len, spatial_tokens.device, inputs_embeds.dtype, text_pad_mask
        )
        
        if labels is not None:
            vis_labels = torch.full((batch_size, self.num_patches), -100, dtype=labels.dtype, device=labels.device)
            combined_labels = torch.cat([vis_labels, labels], dim=1)
        else:
            combined_labels = None
            
        outputs = self.qwen(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask_4d,
            labels=combined_labels
        )
        return outputs.loss, cls_logits

model = MultiTaskQwenModel(qwen_lora_model).to(device)

# -------------------------------------------------------------------------
# Dataset & DataLoader (Shape-Safe Memory Mapping with float16 Spatial Features)
# -------------------------------------------------------------------------
def find_artifact_file(target_filename, fallback_folder):
    for s_dir in search_dirs:
        if not s_dir.exists():
            continue
        for root, _, files in os.walk(s_dir):
            if target_filename in files:
                return os.path.join(root, target_filename)
    return os.path.join(fallback_folder, target_filename)

class FeaturesDataset(Dataset):
    def __init__(self, df, split_name, total_split_len, tokenizer, max_seq_len=160, max_rag_tokens=40):
        self.df = df
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.max_rag_tokens = max_rag_tokens
        
        spatial_file = find_artifact_file(f"features_spatial_{split_name}.dat", features_base)
        mean_file = find_artifact_file(f"features_mean_{split_name}.dat", features_base)
        
        print(f"Loading Memory-Mapped Features for [{split_name}]:", flush=True)
        print(f"  Spatial (float16): {spatial_file}", flush=True)
        print(f"  Mean    (float32): {mean_file}", flush=True)
        
        # Spatial features were generated as float16 in Data Prep to respect Kaggle disk limits
        self.spatial_mmap = np.memmap(spatial_file, dtype="float16", mode="r", shape=(total_split_len, 128, 768))
        self.mean_mmap = np.memmap(mean_file, dtype="float32", mode="r", shape=(total_split_len, 768))
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        orig_idx = int(row["orig_split_idx"])
        spatial_token = torch.from_numpy(np.array(self.spatial_mmap[orig_idx]))
        mean_feat = torch.from_numpy(np.array(self.mean_mmap[orig_idx]))
        
        # Ground Truth CheXpert multi-label vector (from master CSV columns)
        cls_labels = torch.tensor([row[c] for c in CHEXPERT_COLS], dtype=torch.float)
        
        retrieved_raw = row["retrieved_report"]
        if pd.isna(retrieved_raw) or str(retrieved_raw).strip().lower() in ["", "nan", "none"]:
            retrieved_clean = "Normal chest radiograph with no acute cardiopulmonary abnormality."
        else:
            retrieved_clean = str(retrieved_raw).strip()
            
        # Truncate RAG context cleanly
        rag_tokens = self.tokenizer.encode(retrieved_clean, add_special_tokens=False)[:self.max_rag_tokens]
        retrieved_clean = self.tokenizer.decode(rag_tokens, skip_special_tokens=True)
            
        gt_report = str(row["report"]).strip() if pd.notna(row["report"]) else "Normal chest radiograph."
        
        # -----------------------------------------------------------------
        # Model 5 Pure Completion Format: Eliminates ChatML Style Mismatch
        # -----------------------------------------------------------------
        prefix_str = f"Context Findings: {retrieved_clean}\nFindings Impression:"
        full_str = f"{prefix_str} {gt_report}"
        
        prefix_tokens = self.tokenizer.encode(prefix_str, add_special_tokens=False)
        prefix_len = len(prefix_tokens)
        
        enc = self.tokenizer(
            full_str,
            truncation=True,
            max_length=self.max_seq_len,
            padding="max_length",
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        labels[:min(prefix_len, self.max_seq_len)] = -100  # Only compute text loss on ground truth report
        
        return spatial_token, mean_feat, input_ids, labels, attention_mask, cls_labels

train_ds = FeaturesDataset(train_sampled_df, "train", len(train_df), tokenizer, max_seq_len=160)
val_ds = FeaturesDataset(val_df, "val", len(val_df), tokenizer, max_seq_len=160)

# Multi-processing num_workers=0 avoids race conditions with memory mapped files
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=False)

# -------------------------------------------------------------------------
# Asymmetric Loss (ASL) for Multi-Label Clinical Pathology Classification
# -------------------------------------------------------------------------
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-7, pos_weight=None):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        logits = logits.float()
        targets = targets.float()
        probs = torch.sigmoid(logits).clamp(min=self.eps, max=1.0 - self.eps)
        
        targets_pos = targets
        targets_neg = 1.0 - targets

        probs_neg = probs
        if self.clip is not None and self.clip > 0:
            probs_neg = (probs_neg - self.clip).clamp(min=0.0, max=1.0 - self.eps)

        loss_pos = targets_pos * ((1.0 - probs) ** self.gamma_pos) * torch.log(probs)
        loss_neg = targets_neg * (probs_neg ** self.gamma_neg) * torch.log(1.0 - probs_neg)

        if self.pos_weight is not None:
            loss_pos = loss_pos * self.pos_weight

        loss = -(loss_pos + loss_neg)
        return loss.mean()

criterion_cls = AsymmetricLoss(gamma_neg=4.0, gamma_pos=1.0, clip=0.05, pos_weight=pos_weights_tensor)

# -------------------------------------------------------------------------
# Optimizer & Learning Rate Scheduling
# -------------------------------------------------------------------------
trainable_params = [
    {"params": model.visual_projection.parameters(), "lr": 3e-4, "weight_decay": 0.01},
    {"params": model.classifier.parameters(), "lr": 1e-4, "weight_decay": 0.01},
    {"params": [p for p in model.qwen.parameters() if p.requires_grad], "lr": 5e-5, "weight_decay": 0.01}
]
optimizer = torch.optim.AdamW(trainable_params)

GLOBAL_TOTAL_EPOCHS = 4
EPOCHS_PER_SESSION = 2
accumulation_steps = 4
steps_per_epoch = len(train_loader) // accumulation_steps
total_training_steps = steps_per_epoch * GLOBAL_TOTAL_EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.05 * total_training_steps),
    num_training_steps=total_training_steps
)

# -------------------------------------------------------------------------
# Checkpoint Auto-Resume Mechanism (Cross-Session Seamless Chaining)
# -------------------------------------------------------------------------
start_epoch = 1
best_val_loss = float("inf")

prev_ckpts = glob.glob("/kaggle/input/**/model5_qwen_latest.pt", recursive=True) + \
             glob.glob("/kaggle/input/**/model5_qwen_best.pt", recursive=True) + \
             glob.glob("/kaggle/working/model5_qwen_latest.pt") + \
             glob.glob("/kaggle/working/model5_qwen_best.pt")

resume_ckpt_path = None
if prev_ckpts:
    best_candidate = None
    highest_epoch = -1
    for p in prev_ckpts:
        try:
            temp_ckpt = torch.load(p, map_location="cpu")
            ep = temp_ckpt.get("epoch", 1)
            is_best = "best" in os.path.basename(p).lower()
            if (ep > highest_epoch) or (ep == highest_epoch and is_best):
                highest_epoch = ep
                best_candidate = p
        except Exception:
            pass
    resume_ckpt_path = best_candidate

if resume_ckpt_path and os.path.exists(resume_ckpt_path):
    print(f"Resuming training from checkpoint: {resume_ckpt_path}", flush=True)
    ckpt = torch.load(resume_ckpt_path, map_location=device)
    model.visual_projection.load_state_dict(ckpt["visual_projection"])
    model.classifier.load_state_dict(ckpt["classifier"])
    set_peft_model_state_dict(qwen_lora_model, ckpt["lora_state"])
    if "optimizer" in ckpt:
        try:
            optimizer.load_state_dict(ckpt["optimizer"])
        except Exception as e:
            print(f"Warning: could not restore optimizer state ({e})", flush=True)
    if "scheduler" in ckpt:
        try:
            scheduler.load_state_dict(ckpt["scheduler"])
        except Exception as e:
            print(f"Warning: could not restore scheduler state ({e})", flush=True)
    start_epoch = ckpt.get("epoch", 1) + 1
    best_val_loss = ckpt.get("val_loss", float("inf"))
    print(f"Resumed state loaded! Starting at Epoch {start_epoch} (Prior Best Val Loss: {best_val_loss:.4f})", flush=True)
else:
    print("No prior checkpoint found. Initializing training from scratch (Epoch 1).", flush=True)

end_epoch = start_epoch + EPOCHS_PER_SESSION - 1
print(f"Session Execution Target: Epoch {start_epoch} to Epoch {end_epoch} (Max Global: {GLOBAL_TOTAL_EPOCHS})", flush=True)

# -------------------------------------------------------------------------
# Training Loop
# -------------------------------------------------------------------------
scaler = torch.amp.GradScaler("cuda")
training_history = []
prior_history_files = glob.glob("/kaggle/input/**/training_history.json", recursive=True)
if prior_history_files and os.path.exists(prior_history_files[0]):
    try:
        with open(prior_history_files[0]) as f:
            training_history = json.load(f)
            print(f"Loaded {len(training_history)} prior epoch history records.", flush=True)
    except Exception:
        pass

for epoch in range(start_epoch, end_epoch + 1):
    epoch_start_time = time.time()
    model.train()
    running_joint_loss, running_text_loss, running_cls_loss = 0.0, 0.0, 0.0
    optimizer.zero_grad()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{end_epoch} [Train]")
    for step, (spatial_t, mean_f, input_ids, labels, attn_mask, cls_targets) in enumerate(pbar):
        spatial_t = spatial_t.to(device)
        mean_f = mean_f.to(device)
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        attn_mask = attn_mask.to(device)
        cls_targets = cls_targets.to(device)
        
        with torch.amp.autocast("cuda"):
            loss_text, cls_logits = model(spatial_t, mean_f, input_ids, labels=labels, text_pad_mask=attn_mask)
            loss_cls = criterion_cls(cls_logits, cls_targets)
            joint_loss = (loss_text + 1.0 * loss_cls) / accumulation_steps
            
        scaler.scale(joint_loss).backward()
        
        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            all_trainable = [p for p in model.parameters() if p.requires_grad]
            torch.nn.utils.clip_grad_norm_(all_trainable, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
        running_joint_loss += joint_loss.item() * accumulation_steps
        running_text_loss += loss_text.item()
        running_cls_loss += loss_cls.item()
        
        if (step + 1) % 10 == 0 or (step + 1) == len(train_loader):
            current_lr = scheduler.get_last_lr()[0]
            pbar.set_postfix({
                "Joint": f"{joint_loss.item() * accumulation_steps:.4f}",
                "Text": f"{loss_text.item():.4f}",
                "Cls": f"{loss_cls.item():.4f}",
                "LR": f"{current_lr:.2e}"
            })
            
    epoch_train_joint = running_joint_loss / len(train_loader)
    epoch_train_text = running_text_loss / len(train_loader)
    epoch_train_cls = running_cls_loss / len(train_loader)
    
    # ---------------------------------------------------------------------
    # Validation Phase
    # ---------------------------------------------------------------------
    model.eval()
    val_joint_loss, val_text_loss, val_cls_loss = 0.0, 0.0, 0.0
    all_val_probs, all_val_targets = [], []
    
    with torch.no_grad():
        for spatial_t, mean_f, input_ids, labels, attn_mask, cls_targets in tqdm(val_loader, desc=f"Epoch {epoch}/{end_epoch} [Val]"):
            spatial_t = spatial_t.to(device)
            mean_f = mean_f.to(device)
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            attn_mask = attn_mask.to(device)
            cls_targets = cls_targets.to(device)
            
            with torch.amp.autocast("cuda"):
                loss_text, cls_logits = model(spatial_t, mean_f, input_ids, labels=labels, text_pad_mask=attn_mask)
                loss_cls = criterion_cls(cls_logits, cls_targets)
                joint_loss = loss_text + 1.0 * loss_cls
                
            val_joint_loss += joint_loss.item()
            val_text_loss += loss_text.item()
            val_cls_loss += loss_cls.item()
            
            probs = torch.sigmoid(cls_logits.float()).cpu().numpy()
            all_val_probs.append(probs)
            all_val_targets.append(cls_targets.cpu().numpy())
            
    epoch_val_joint = val_joint_loss / len(val_loader)
    epoch_val_text = val_text_loss / len(val_loader)
    epoch_val_cls = val_cls_loss / len(val_loader)
    
    val_probs_arr = np.vstack(all_val_probs)
    val_targets_arr = np.vstack(all_val_targets)
    
    val_preds_binary = (val_probs_arr >= 0.5).astype(int)
    val_macro_f1 = f1_score(val_targets_arr, val_preds_binary, average="macro", zero_division=0)
    
    elapsed_epoch = (time.time() - epoch_start_time) / 60.0
    print(f"\n--- Epoch {epoch} Summary ({elapsed_epoch:.1f}m) ---", flush=True)
    print(f"Train Loss : Joint={epoch_train_joint:.4f} | Text={epoch_train_text:.4f} | Cls={epoch_train_cls:.4f}", flush=True)
    print(f"Val Loss   : Joint={epoch_val_joint:.4f} | Text={epoch_val_text:.4f} | Cls={epoch_val_cls:.4f}", flush=True)
    print(f"Val Metric : Macro F1 (at 0.5)={val_macro_f1:.4f}", flush=True)
    print("-" * 55, flush=True)
    
    # ---------------------------------------------------------------------
    # Save Checkpoints
    # ---------------------------------------------------------------------
    ckpt_state = {
        "visual_projection": model.visual_projection.state_dict(),
        "classifier": model.classifier.state_dict(),
        "lora_state": {k: v.cpu() for k, v in get_peft_model_state_dict(qwen_lora_model).items()},
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": epoch,
        "val_loss": epoch_val_joint
    }
    
    torch.save(ckpt_state, "/kaggle/working/model5_qwen_latest.pt")
    print(f"Saved latest checkpoint to /kaggle/working/model5_qwen_latest.pt", flush=True)
    
    if epoch_val_joint < best_val_loss:
        best_val_loss = epoch_val_joint
        torch.save(ckpt_state, "/kaggle/working/model5_qwen_best.pt")
        print(f"[*] New best validation loss achieved ({best_val_loss:.4f})! Saved model5_qwen_best.pt", flush=True)
        
    training_history.append({
        "epoch": epoch,
        "train_joint": epoch_train_joint, "train_text": epoch_train_text, "train_cls": epoch_train_cls,
        "val_joint": epoch_val_joint, "val_text": epoch_val_text, "val_cls": epoch_val_cls,
        "val_macro_f1": val_macro_f1
    })

with open("/kaggle/working/training_history.json", "w") as f:
    json.dump(training_history, f, indent=2)

# Generate and save Learning Curves
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = [h["epoch"] for h in training_history]
ax1.plot(epochs_range, [h["train_joint"] for h in training_history], marker="o", label="Train Joint Loss")
ax1.plot(epochs_range, [h["val_joint"] for h in training_history], marker="s", label="Val Joint Loss")
ax1.set_title("Model 5 Joint Loss Progression", fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(epochs_range, [h["train_text"] for h in training_history], marker="o", label="Train Text Loss")
ax2.plot(epochs_range, [h["val_text"] for h in training_history], marker="s", label="Val Text Loss")
ax2.set_title("Model 5 Text Completion Loss Progression", fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=300)
plt.close()
print("Learning curves plot saved to /kaggle/working/learning_curves.png", flush=True)

# -------------------------------------------------------------------------
# Dynamic Optimal Threshold Computation on Final Validation Predictions
# -------------------------------------------------------------------------
print("\nComputing Class-Specific Optimal Decision Thresholds on Validation Cohort...", flush=True)
optimal_thresholds = {}
for i, col in enumerate(CHEXPERT_COLS):
    best_f1, best_thresh = 0.0, 0.5
    for thresh in np.arange(0.1, 0.9, 0.02):
        preds = (val_probs_arr[:, i] >= thresh).astype(int)
        f1 = f1_score(val_targets_arr[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    optimal_thresholds[col] = float(np.round(best_thresh, 3))
    print(f"  {col:<26} | Optimal Thresh: {best_thresh:.2f} | Val F1: {best_f1:.4f}", flush=True)

with open("/kaggle/working/optimal_thresholds.json", "w") as f:
    json.dump(optimal_thresholds, f, indent=2)

print("\nOptimal Thresholds successfully saved to /kaggle/working/optimal_thresholds.json", flush=True)
print("Model 5 Training Session Completed Successfully!", flush=True)
